In [ ]:
import pandas as pd
from pathlib import Path
import json

path = Path("../data/results")

df_all = pd.concat(
    [
        pd.json_normalize(json.load(open(file))[
                          "artifacts"]).assign(repo=file.stem)
        for file in path.glob("*-sbom.json")
    ],
    ignore_index=True
)

df_all = df_all.rename(columns={
    "name": "package"
})

df_all.head()

,id,package,version,type,foundBy,locations,licenses,language,cpes,purl,...,metadata.dependencies.@esbuild/linux-riscv64,metadata.dependencies.@esbuild/linux-s390x,metadata.dependencies.@esbuild/linux-x64,metadata.dependencies.@esbuild/netbsd-x64,metadata.dependencies.@esbuild/openbsd-x64,metadata.dependencies.@esbuild/sunos-x64,metadata.dependencies.@esbuild/win32-arm64,metadata.dependencies.@esbuild/win32-ia32,metadata.dependencies.@esbuild/win32-x64,metadata.dependencies.flowise-embed-react
0,0283ff70c757864e,@aashutoshrathi/word-wrap,1.2.6,npm,javascript-lock-cataloger,"[{'path': '/pnpm-lock.yaml', 'accessPath': '/p...",[],javascript,[{'cpe': 'cpe:2.3:a:\@aashutoshrathi\/word-wra...,pkg:npm/%40aashutoshrathi/word-wrap@1.2.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,149ca2f5cd7a0935,@adobe/css-tools,4.3.3,npm,javascript-lock-cataloger,"[{'path': '/pnpm-lock.yaml', 'accessPath': '/p...",[],javascript,[{'cpe': 'cpe:2.3:a:adobe:css-tools:4.3.3:*:*:...,pkg:npm/%40adobe/css-tools@4.3.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,b332513e51df82a8,@adobe/css-tools,4.4.4,npm,javascript-lock-cataloger,"[{'path': '/pnpm-lock.yaml', 'accessPath': '/p...",[],javascript,[{'cpe': 'cpe:2.3:a:adobe:css-tools:4.4.4:*:*:...,pkg:npm/%40adobe/css-tools@4.4.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,bdcc0d397ed9a86c,@ai-sdk/openai,3.0.2,npm,javascript-lock-cataloger,"[{'path': '/pnpm-lock.yaml', 'accessPath': '/p...",[],javascript,[{'cpe': 'cpe:2.3:a:\@ai-sdk\/openai:\@ai-sdk\...,pkg:npm/%40ai-sdk/openai@3.0.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8dd563c7984906f1,@ai-sdk/provider,0.0.10,npm,javascript-lock-cataloger,"[{'path': '/pnpm-lock.yaml', 'accessPath': '/p...",[],javascript,[{'cpe': 'cpe:2.3:a:\@ai-sdk\/provider:\@ai-sd...,pkg:npm/%40ai-sdk/provider@0.0.10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
deps_cols = [col for col in df_all.columns if col.startswith(
    "metadata.dependencies.")]

deps_long = df_all.melt(
    id_vars=["repo", "package", "version"],
    value_vars=deps_cols,
    var_name="dependency",
    value_name="dep_version"
).dropna()

deps_long["dependency"] = deps_long["dependency"].str.replace(
    "metadata.dependencies.", "")

In [4]:
df_all['repo'].value_counts()

repo
Flowise-sbom              4540
FlowiseEmbedReact-sbom     604
FlowiseChatEmbed-sbom      380
Name: count, dtype: int64

In [5]:
df_all['licenses'].explode().value_counts().head(10)

licenses
{'value': 'MIT', 'spdxExpression': 'MIT', 'type': 'declared', 'urls': [], 'locations': [{'path': '/package-lock.json', 'accessPath': '/package-lock.json', 'annotations': {'evidence': 'primary'}}]}                                                                              158
{'value': 'MIT', 'spdxExpression': 'MIT', 'type': 'declared', 'urls': [], 'locations': [{'path': '/packages/agentflow/examples/package-lock.json', 'accessPath': '/packages/agentflow/examples/package-lock.json', 'annotations': {'evidence': 'primary'}}]}                      120
{'value': 'MIT', 'spdxExpression': 'MIT', 'type': 'declared', 'urls': [], 'locations': [{'path': '/packages/observe/examples/package-lock.json', 'accessPath': '/packages/observe/examples/package-lock.json', 'annotations': {'evidence': 'primary'}}]}                           79
{'value': 'ISC', 'spdxExpression': 'ISC', 'type': 'declared', 'urls': [], 'locations': [{'path': '/packages/agentflow/examples/package-lock.json', 'accessPat

In [7]:
summary = {
    "total_repos": df_all["repo"].nunique(),
    "total_components": len(df_all),
}
summary

{'total_repos': 3, 'total_components': 5524}

In [8]:
most_used = (
    df_all.groupby("package")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

most_used.head(20)

KeyError: 'package'